## Leitura da Bronze

In [0]:
# Leitura da tabela bronze
df_bronze = spark.read.table("acidentes.bronze.acidentes_2025")

display(df_bronze)

In [0]:
df_bronze.printSchema()

## 📊 Análise exploratória

In [0]:
from pyspark.sql import functions as F

print("Linhas:", df_bronze.count())
print("Colunas:", len(df_bronze.columns))

display(df_bronze.summary())

## Valores duplicados


In [0]:
# Colunas de metadados de ingestão (não considerar para duplicados)
cols_metadados = ["ingestion_timestamp", "ingestion_date", "ingestion_id", "source_table"]

# Colunas de negócio (considerar para duplicados)
cols_negocio = [c for c in df_bronze.columns if c not in cols_metadados]

total = df_bronze.count()
sem_duplicados = df_bronze.dropDuplicates(subset=cols_negocio).count()

print("Total de registros:", total)
print("Registros únicos:", sem_duplicados)
print("Duplicados:", total - sem_duplicados)

## Valores nulos ou inválidos

In [0]:
# Nulos
display(
    df_bronze.select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in df_bronze.columns
    ])
)

In [0]:
valores_invalidos = ["", " ", "NA", "N/A", "NULL", "null", "NaN"]

# Conta valores inválidos por coluna
invalid_counts = {
    c: df_bronze.filter(
        (F.col(c).isNull()) | (F.trim(F.col(c)).isin(valores_invalidos))
    ).count()
    for c in df_bronze.columns
}

# Exibe apenas colunas com valores inválidos (>0)
for c, count in invalid_counts.items():
    if count > 0:
        print(f"{c}: {count}")

In [0]:
display(df_bronze.select("classificacao_acidente").distinct())

In [0]:
# Analise de colunas com valores inválidos
cols = ["classificacao_acidente", "regional", "delegacia", "uop"]

print("🔎 Linhas com valores inválidos em classificacao_acidente")

display(
    df_bronze.filter(
        F.col("classificacao_acidente").isNull() |
        F.trim(F.col("classificacao_acidente").cast("string")).isin(valores_invalidos)
    )
)

In [0]:
# Corrigir valores inválidos de classificacao_acidente baseado em mortos/feridos
df_bronze = df_bronze.withColumn(
    "classificacao_acidente",
    F.when(
        (F.col("classificacao_acidente").isNull()) |
        (F.trim(F.col("classificacao_acidente")).isin(valores_invalidos)),
        F.when(F.col("mortos") > 0, "Com Vítimas Fatais")
         .when(F.col("feridos") > 0, "Com Vítimas Feridas")
         .otherwise("Sem Vítimas")
    ).otherwise(F.col("classificacao_acidente"))
)

print("✅ Valores de classificacao_acidente corrigidos")
display(df_bronze.select("classificacao_acidente").distinct())

In [0]:
# Não fazem sentido para a analise as colunas regional, delegacia, uop, latitude, longitude e metadados de governança
colunas_remover = ["regional", "delegacia", "uop", "latitude", "longitude", "ingestion_timestamp", "ingestion_date", "ingestion_id", "source_table"]
df = df_bronze.select([c for c in df_bronze.columns if c not in colunas_remover])
display(df)

## Analise individual de cada coluna

In [0]:
df.columns